# Spoken announcement intelligibility (Whisper)

Transcribes a set of synthesised bus announcements with Whisper so it can be scored
locally with `scripts/score_tts_transcripts.py`. This notebook only transcribes;
it does not score. No Google Drive needed.

1. On Windows, build the set (see `evaluation/tts/README.md`). Each set is written as
   `.runtime/tts-eval/<set>.zip` and as 10 MB parts in `.runtime/tts-eval/<set>-parts/`
   (sets: `announcements-v2`, `service-wording-v3`).
2. Connect to a Colab **GPU** runtime (T4 is enough). In VS Code: Select Kernel > Colab >
   a new server with a GPU. In the browser: Runtime > Change runtime type > GPU.
3. Upload the audio. In VS Code: right-click the `<set>-parts` folder in the Explorer >
   Upload to Colab (a single large upload fails there). In the browser: drag the zip into
   the Files panel, or run the cells and pick it in the upload box.
4. Run all cells. The notebook uses the set uploaded most recently (or set `RUN` in the
   second code cell). `transcripts.csv` is written to `/content`. Download it (VS Code:
   Colab sidebar > Contents > right-click > Download; the browser downloads it
   automatically), put it in `evaluation/tts/<set>/` and run the scoring script.

Whisper gets no prompt or vocabulary hints, so it hears the audio the way a
first-time listener would (as far as a speech recogniser can stand in for one).


In [5]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip -q install faster-whisper==1.1.1 soundfile nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import ctypes, glob, importlib.util, os
import ctranslate2
if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError('No GPU on this runtime. Connect to a Colab GPU server (T4) and run again; '
                       'large-v3 on CPU would take hours.')

# faster-whisper's CTranslate2 is built for CUDA 12 (libcublas.so.12, cuDNN 9). Newer Colab images ship a
# different CUDA, so load the CUDA 12 libraries from the pip wheels above before the model needs them.
# Libraries already loaded this way are found by name when CTranslate2 asks for them.
def cuda12_libraries():
    for package, pattern in [('nvidia.cuda_runtime', 'libcudart.so.12'), ('nvidia.cublas', 'libcublas*.so.12'),
                             ('nvidia.cudnn', 'libcudnn*.so.9')]:
        spec = importlib.util.find_spec(package)
        for root in (spec.submodule_search_locations if spec else []):
            yield from sorted(glob.glob(os.path.join(root, 'lib', pattern)))

pending, loaded = list(cuda12_libraries()), []
while pending:  # cuDNN parts depend on each other: retry until nothing more loads
    failed = []
    for lib in pending:
        try:
            ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
            loaded.append(os.path.basename(lib))
        except OSError:
            failed.append(lib)
    if len(failed) == len(pending):
        break
    pending = failed
print('Loaded CUDA 12 libraries:', ', '.join(loaded) or 'none')
if pending:
    print('Could not load:', ', '.join(os.path.basename(lib) for lib in pending))
if not any(name.startswith('libcublas.so') for name in loaded):
    raise RuntimeError('libcublas.so.12 did not load; transcription will fail on the GPU')

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Loaded CUDA 12 libraries: libcudart.so.12, libcublas.so.12, libcublasLt.so.12, libcudnn.so.9, libcudnn_adv.so.9, libcudnn_cnn.so.9, libcudnn_engines_precompiled.so.9, libcudnn_engines_runtime_compiled.so.9, libcudnn_engines_tensor_ir.so.9, libcudnn_ext.so.9, libcudnn_graph.so.9, libcudnn_heuristic.so.9, libcudnn_ops.so.9


In [6]:
import hashlib, zipfile, pathlib
RUN = None  # None: the set uploaded most recently; or name one, e.g. 'announcements-v2'
CONTENT = pathlib.Path('/content')
TTS = CONTENT / 'tts'  # where uploads are joined and extracted
TTS.mkdir(parents=True, exist_ok=True)

def uploaded(pattern):
    return [p for p in CONTENT.rglob(pattern) if TTS not in p.parents]

# 1. A zip uploaded in parts (VS Code: Upload to Colab on the <set>-parts folder).
# 2. A whole zip (browser Files panel); a half-finished upload is not a valid zip.
parts_by_run = {}
for part in uploaded('*.zip.part*'):
    parts_by_run.setdefault(part.name.split('.zip.part')[0], []).append(part)
wholes = {p.name[:-len('.zip')]: p for p in uploaded('*.zip') if zipfile.is_zipfile(p)}
if RUN is None:
    newest = {run: p.stat().st_mtime for run, p in wholes.items()}
    newest.update({run: max(p.stat().st_mtime for p in ps) for run, ps in parts_by_run.items()})
    if newest:
        RUN = max(newest, key=newest.get)
        if len(newest) > 1:
            print('Uploaded sets:', ', '.join(sorted(newest)), '- using the newest; set RUN to choose')

if RUN in parts_by_run:
    parts = sorted(parts_by_run[RUN])
    bundle = TTS / f'{RUN}.zip'
    with open(bundle, 'wb') as joined:
        for part in parts:
            joined.write(part.read_bytes())
    checksum = next(iter(uploaded(f'{RUN}.zip.sha256')), None)
    digest = hashlib.sha256(bundle.read_bytes()).hexdigest()
    if checksum and checksum.read_text().strip() != digest:
        raise ValueError(f'Joined {len(parts)} parts, but the checksum does not match: re-upload the parts folder')
    print(f'Joined {len(parts)} parts', '(checksum OK)' if checksum else '(no checksum file)')
elif RUN in wholes:
    bundle = wholes[RUN]
else:
    # 3. Browser only: the upload box does not work inside VS Code.
    from google.colab import files
    uploaded_now = files.upload()
    name = next((n for n in uploaded_now if n.endswith('.zip')), None)
    if name is None:
        raise FileNotFoundError('Upload a <set>-parts folder (or <set>.zip) to the runtime first')
    bundle, RUN = CONTENT / name, name[:-len('.zip')]
WORK = TTS / RUN
print('Set', RUN, 'from', bundle, f'({bundle.stat().st_size / 2**20:.0f} MB)')
with zipfile.ZipFile(bundle) as z:
    z.extractall(WORK)
audio = sorted(WORK.glob('audio/*/*.flac'))
print(len(audio), 'clips;', 'voices:', sorted({p.parent.name for p in audio}))


Uploaded sets: announcements-v2, service-wording-v3 - using the newest; set RUN to choose
Joined 19 parts (checksum OK)
Set service-wording-v3 from /content/tts/service-wording-v3.zip (190 MB)
2748 clips; voices: ['david', 'mark', 'zira']


In [7]:
import csv, time
import soundfile
from faster_whisper import WhisperModel

MODEL = 'large-v3'
model = WhisperModel(MODEL, device='cuda', compute_type='float16')
voices = {'david': 'Microsoft David', 'mark': 'Microsoft Mark', 'zira': 'Microsoft Zira'}

rows = []
started = time.time()
for i, path in enumerate(audio, 1):
    # Read the FLAC directly: faster-whisper's own decoder breaks with newer PyAV on Colab
    # (av.open() lost metadata_errors). The clips are already 16 kHz mono, as Whisper expects.
    samples, rate = soundfile.read(str(path), dtype='float32')
    if rate != 16000 or samples.ndim != 1:
        raise ValueError(f'{path.name}: expected 16 kHz mono, got {rate} Hz, shape {samples.shape}')
    # Deterministic decoding, no prompt, no carry-over between clips.
    segments, _ = model.transcribe(samples, language='en', beam_size=5, temperature=0,
                                   condition_on_previous_text=False, vad_filter=False)
    text = ' '.join(s.text.strip() for s in segments).strip()
    rows.append({'voice': voices.get(path.parent.name, path.parent.name), 'clip_id': path.stem,
                 'transcript': text, 'model': f'faster-whisper {MODEL}'})
    if i % 200 == 0:
        print(f'{i}/{len(audio)} clips, {time.time() - started:.0f} s')

out = pathlib.Path('/content/transcripts.csv')
with open(out, 'w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print('Wrote', out, f'({len(rows)} rows, {time.time() - started:.0f} s)')

try:  # automatic download works in the browser, not in VS Code
    from google.colab import files
    files.download(str(out))
except Exception:
    print('Download it from the Colab sidebar: Contents > transcripts.csv > Download')
print(f'Save it as evaluation/tts/{RUN}/transcripts.csv, then run the scoring script.')

200/2748 clips, 115 s
400/2748 clips, 230 s
600/2748 clips, 347 s
800/2748 clips, 462 s
1000/2748 clips, 579 s
1200/2748 clips, 694 s
1400/2748 clips, 809 s
1600/2748 clips, 926 s
1800/2748 clips, 1041 s
2000/2748 clips, 1155 s
2200/2748 clips, 1269 s
2400/2748 clips, 1384 s
2600/2748 clips, 1498 s
Wrote /content/transcripts.csv (2748 rows, 1582 s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Save it as evaluation/tts/service-wording-v3/transcripts.csv, then run the scoring script.


In [8]:
# Spot-check a few transcripts next to their text.
import pandas as pd
clips = pd.read_csv(WORK / 'clips.csv')
seen = pd.DataFrame(rows).merge(clips, on='clip_id')
pd.set_option('display.max_colwidth', 120)
seen.sample(8, random_state=0)[['voice', 'text', 'transcript']]

,voice,text,transcript
1801,Microsoft Mark,Bus 2 6 8 A to Ang Mo Kio Avenue 5.,Bus 2688 to Ang Mokio Avenue 5.
1590,Microsoft Mark,"Bus 1 7 A to Bedok North Avenue 4, S B S T Bedok North Depot.","Bus 178 to Beddock North Avenue 4, SBST Beddock North Depot."
2382,Microsoft Zira,Bus 2 4 0 M to Boon Lay M R T station.,Bus 240M to Boon Lay MRT Station
2447,Microsoft Zira,"Bus 1 3 1 A, to Shenton Way.",Bus 131A to Shenton Way
1147,Microsoft Mark,Bus 2 9 3 T to Tampines Street 71 and Avenue 7.,Bus 293T to Tampines Street 71 and Avenue 7.
1231,Microsoft Mark,"Bus 138A, to Singapore Zoo.",Bus 138A to Singapore Zoo.
1005,Microsoft Mark,"Bus 3 3 B to Old Airport Road, Block 2 2.","Bus 33B to Old Airport Road, Block 22."
195,Microsoft David,Bus 159B to Ang Mo Kio Avenue 5.,Bus 159B to Ang Mokio Avenue 5
